In [1]:
%pip install fastapi uvicorn xgboost scikit-learn pandas requests ephem joblib nest-asyncio

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 1.9 MB/s eta 0:00:55
   ---------------------------------------- 1.0/101.7 MB 1.9 MB/s eta 0:00:53
    --------------------------------------- 1.3/101.7 MB 2.0 MB/s eta 0:00:51
    --------------------------------------- 1.8/101.7 MB 2.0 MB/s eta 0:00:51
    --------------------------------------- 2.1/101.7 MB 2.0 MB/s eta 0:00:51
   - -------------------------------------- 2.6/101.7 MB 2.0 MB/s eta 0:00:50
   - -------------------------------------- 2.9/101.7 MB 1.9 MB/s eta 0:00:53
   - -------------------------------------- 3.1/101.7 MB 1.9 MB/s eta 0:00:52
   - -------------------------------------- 3.7/101.7 MB 1.9 MB/s eta 0:00:53
   - -------------------------------------- 3.9/101.7 MB 1.8 MB/s eta 0:00:53
   - -------------------------------------- 4.2/101.7 MB 1.8 MB/s eta 0:00:54



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import requests
from datetime import datetime

import joblib
import ephem
import pandas as pd
from datetime import datetime

In [2]:
CITIES = {
    "Banda Aceh":     {"lat": 5.55,   "lon": 95.32,  "tz": "Asia/Jakarta"},
    "Medan":          {"lat": 3.59,   "lon": 98.67,  "tz": "Asia/Jakarta"},
    "Pekanbaru":      {"lat": 0.51,   "lon": 101.44, "tz": "Asia/Jakarta"},
    "Batam":          {"lat": 1.13,   "lon": 104.05, "tz": "Asia/Jakarta"},
    "Padang":         {"lat": -0.94,  "lon": 100.35, "tz": "Asia/Jakarta"},
    "Jambi":          {"lat": -1.61,  "lon": 103.61, "tz": "Asia/Jakarta"},
    "Palembang":      {"lat": -2.99,  "lon": 104.75, "tz": "Asia/Jakarta"},
    "Bengkulu":       {"lat": -3.80,  "lon": 102.27, "tz": "Asia/Jakarta"},
    "Bandar Lampung": {"lat": -5.45,  "lon": 105.26, "tz": "Asia/Jakarta"},
    "Pangkal Pinang": {"lat": -2.13,  "lon": 106.12, "tz": "Asia/Jakarta"},
    "Tanjung Pinang": {"lat": 0.92,   "lon": 104.46, "tz": "Asia/Jakarta"},
    "Jakarta":        {"lat": -6.21,  "lon": 106.85, "tz": "Asia/Jakarta"},
    "Bogor":          {"lat": -6.60,  "lon": 106.80, "tz": "Asia/Jakarta"},
    "Bandung":        {"lat": -6.91,  "lon": 107.61, "tz": "Asia/Jakarta"},
    "Serang":         {"lat": -6.12,  "lon": 106.15, "tz": "Asia/Jakarta"},
    "Semarang":       {"lat": -6.97,  "lon": 110.42, "tz": "Asia/Jakarta"},
    "Yogyakarta":     {"lat": -7.80,  "lon": 110.36, "tz": "Asia/Jakarta"},
    "Solo":           {"lat": -7.57,  "lon": 110.83, "tz": "Asia/Jakarta"},
    "Surabaya":       {"lat": -7.25,  "lon": 112.75, "tz": "Asia/Jakarta"},
    "Malang":         {"lat": -7.98,  "lon": 112.63, "tz": "Asia/Jakarta"},
    "Madiun":         {"lat": -7.63,  "lon": 111.52, "tz": "Asia/Jakarta"},
    "Kediri":         {"lat": -7.82,  "lon": 112.01, "tz": "Asia/Jakarta"},
    "Pontianak":      {"lat": -0.02,  "lon": 109.33, "tz": "Asia/Jakarta"},
    "Palangkaraya":   {"lat": -2.21,  "lon": 113.92, "tz": "Asia/Jakarta"},
    "Banjarmasin":    {"lat": -3.32,  "lon": 114.59, "tz": "Asia/Makassar"},
    "Balikpapan":     {"lat": -1.27,  "lon": 116.83, "tz": "Asia/Makassar"},
    "Samarinda":      {"lat": -0.50,  "lon": 117.15, "tz": "Asia/Makassar"},
    "Tanjung Selor":  {"lat": 2.84,   "lon": 117.37, "tz": "Asia/Makassar"},
    "Nusantara (IKN)":{"lat": -1.10,  "lon": 116.71, "tz": "Asia/Makassar"},
    "Makassar":       {"lat": -5.14,  "lon": 119.41, "tz": "Asia/Makassar"},
    "Palu":           {"lat": -0.90,  "lon": 119.87, "tz": "Asia/Makassar"},
    "Kendari":        {"lat": -3.97,  "lon": 122.51, "tz": "Asia/Makassar"},
    "Mamuju":         {"lat": -2.67,  "lon": 118.89, "tz": "Asia/Makassar"},
    "Gorontalo":      {"lat": 0.54,   "lon": 123.06, "tz": "Asia/Makassar"},
    "Manado":         {"lat": 1.47,   "lon": 124.84, "tz": "Asia/Makassar"},
    "Denpasar":       {"lat": -8.65,  "lon": 115.22, "tz": "Asia/Makassar"},
    "Mataram":        {"lat": -8.58,  "lon": 116.10, "tz": "Asia/Makassar"},
    "Kupang":         {"lat": -10.17, "lon": 123.58, "tz": "Asia/Makassar"},
    "Ambon":          {"lat": -3.69,  "lon": 128.18, "tz": "Asia/Jayapura"},
    "Sofifi":         {"lat": 0.74,   "lon": 127.56, "tz": "Asia/Jayapura"},
    "Jayapura":       {"lat": -2.53,  "lon": 140.72, "tz": "Asia/Jayapura"},
    "Manokwari":      {"lat": -0.86,  "lon": 134.08, "tz": "Asia/Jayapura"},
    "Sorong":         {"lat": -0.88,  "lon": 131.25, "tz": "Asia/Jayapura"},
    "Merauke":        {"lat": -8.49,  "lon": 140.40, "tz": "Asia/Jayapura"},
    "Nabire":         {"lat": -3.37,  "lon": 135.50, "tz": "Asia/Jayapura"},
}

def fetch_forecast(city_name: str) -> list[dict]:
    city = CITIES[city_name]
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": city["lat"],
        "longitude": city["lon"],
        "hourly": [
            "cloudcover",
            "relativehumidity_2m",
            "windspeed_10m",
            "precipitation"
        ],
        "forecast_days": 7,
        "timezone": city["tz"]
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()["hourly"]

    result = []
    for i in range(len(data["time"])):
        result.append({
            "time":                 data["time"][i],
            "cloudcover":          data["cloudcover"][i] or 0,
            "relativehumidity_2m": data["relativehumidity_2m"][i] or 0,
            "windspeed_10m":       data["windspeed_10m"][i] or 0,
            "precipitation":       data["precipitation"][i] or 0,
        })
    return result

In [3]:
model_sky  = joblib.load("astroweather_xgb.pkl")
model_temp = joblib.load("astroweather_temp.pkl")
model_cls  = joblib.load("astroweather_cls.pkl")

WEATHER_LABELS = {0: "Cerah", 1: "Berawan", 2: "Hujan", 3: "Badai"}
WEATHER_ICONS  = {0: "☀️",    1: "⛅",      2: "🌧️",   3: "⛈️"}

def get_moon_phase(date_str: str) -> float:
    dt   = datetime.fromisoformat(date_str)
    moon = ephem.Moon(dt.strftime("%Y/%m/%d"))
    return round(moon.phase, 2)

def predict_all(forecast: list[dict]) -> list[dict]:
    rows = []
    for f in forecast:
        dt    = datetime.fromisoformat(f["time"])
        moon  = get_moon_phase(f["time"])
        rows.append({
            "cloudcover":          f["cloudcover"],
            "relativehumidity_2m": f["relativehumidity_2m"],
            "windspeed_10m":       f["windspeed_10m"],
            "precipitation":       f["precipitation"],
            "moon_phase":          moon,
            "hour":                dt.hour,
            "month":               dt.month,
        })

    df = pd.DataFrame(rows)

    # Prediksi sky score
    sky_scores = model_sky.predict(
        df[["cloudcover","relativehumidity_2m","windspeed_10m","precipitation","moon_phase"]]
    )

    # Prediksi suhu
    temperatures = model_temp.predict(
        df[["cloudcover","relativehumidity_2m","windspeed_10m","precipitation","moon_phase","hour","month"]]
    )

    # Prediksi klasifikasi cuaca
    weather_labels = model_cls.predict(
        df[["cloudcover","relativehumidity_2m","windspeed_10m","moon_phase","hour","month"]]
    )

    result = []
    for i, f in enumerate(forecast):
        result.append({
            "time":          f["time"],
            "sky_score":     round(max(0, min(float(sky_scores[i]),  100)), 2),
            "temperature":   round(float(temperatures[i]), 1),
            "weather_label": WEATHER_LABELS[int(weather_labels[i])],
            "weather_icon":  WEATHER_ICONS[int(weather_labels[i])],
            "cloudcover":    f["cloudcover"],
            "moon_phase":    rows[i]["moon_phase"],
            "precipitation": f["precipitation"],
            "humidity":      f["relativehumidity_2m"],
            "windspeed":     f["windspeed_10m"],
        })
    return result

In [4]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="AstroWeather API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/cities")
def get_cities():
    return {"cities": sorted(CITIES.keys())}

@app.get("/forecast/{city_name}")
def get_forecast(city_name: str):
    if city_name not in CITIES:
        raise HTTPException(status_code=404, detail="Kota tidak ditemukan")
    forecast    = fetch_forecast(city_name)
    predictions = predict_all(forecast)
    return {"city": city_name, "data": predictions}

@app.get("/best-hours/{city_name}")
def get_best_hours(city_name: str):
    if city_name not in CITIES:
        raise HTTPException(status_code=404, detail="Kota tidak ditemukan")
    forecast    = fetch_forecast(city_name)
    predictions = predict_all(forecast)

    night = [p for p in predictions if int(p["time"][11:13]) >= 18 or int(p["time"][11:13]) <= 5]
    top5  = sorted(night, key=lambda x: x["sky_score"], reverse=True)[:5]
    return {"city": city_name, "best_hours": top5}

In [6]:
import nest_asyncio
import uvicorn
import asyncio

nest_asyncio.apply()

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [8792]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:54468 - "GET /cities HTTP/1.1" 200 OK
INFO:     127.0.0.1:59250 - "GET /best-hours/Bandung HTTP/1.1" 200 OK
INFO:     127.0.0.1:56468 - "GET /forecast/Bandung HTTP/1.1" 200 OK
INFO:     127.0.0.1:51519 - "GET /forecast/Kupang HTTP/1.1" 200 OK
INFO:     127.0.0.1:61278 - "GET /best-hours/Kupang HTTP/1.1" 200 OK
INFO:     127.0.0.1:50435 - "GET /forecast/Solo HTTP/1.1" 200 OK
INFO:     127.0.0.1:56049 - "GET /best-hours/Solo HTTP/1.1" 200 OK
INFO:     127.0.0.1:60642 - "GET /best-hours/Banda%20Aceh HTTP/1.1" 200 OK
INFO:     127.0.0.1:52585 - "GET /forecast/Banda%20Aceh HTTP/1.1" 200 OK
INFO:     127.0.0.1:64110 - "GET /best-hours/Bandung HTTP/1.1" 200 OK
INFO:     127.0.0.1:61490 - "GET /forecast/Bandung HTTP/1.1" 200 OK
INFO:     127.0.0.1:54753 - "GET /best-hours/Jakarta HTTP/1.1" 200 OK
INFO:     127.0.0.1:53611 - "GET /forecast/Jakarta HTTP/1.1" 200 OK
INFO:     127.0.0.1:56457 - "GET /best-hours/Kupang HTTP/1.1" 200 OK
INFO:     127.0.0.1:62146 - "GET /forecast/

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [8792]
